# EXP-010b — Test anchor/plain (AffectNet) ở e5/e10/e20/e30
Chỉ test, **không train** (checkpoint EXP-010 và EXP-004 đã có trên Drive). GPU nào cũng được.

Câu hỏi: ở EXP-010, anchor e30 kém plain e5 — do anchor hay do train lâu?
- So **cùng epoch**: anchor AffectNet eE − EXP-004 eE (chỉ đổi ảnh train), anchor eE − plain eE (chỉ đổi loss).
- **Tiêu chí (chọn trước khi xem số):** anchor AffectNet ở epoch E được coi là có ích nếu so với **plain e5 AffectNet** (cấu hình tốt nhất hiện tại), Ensemble Five P0: Mean4 ≥ +0.5 và không dataset nào < −0.5. Có 4 epoch được thử → chỉ tin nếu không phải 1 epoch lẻ loi.

# Cấu hình

In [1]:
SEEDS = [1, 2, 3]
EPOCHS_TEST = [5, 10, 20, 30]
job_of = lambda v, seed: {"plain": f"exp010_affnet_plain_l14_seed{seed}",
                          "anchor": f"exp010_affnet_anchor_crossA_l1_l14_seed{seed}"}[v]
BRANCH = "exp/010-affectnet-train"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp010"
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
import os

# Mount Drive, clone repo, giải nén tập test

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!pip install -q ftfy "setuptools<81"

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (466/466), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 466 (delta 341), reused 400 (delta 276), pack-reused 0 (from 0)
Receiving objects: 100% (466/466), 32.35 MiB | 16.47 MiB/s, done.
Resolving deltas: 100% (341/341), done.
/content/Exp-CLIP
Already on 'exp/010-affectnet-train'
Your branch is up to date with 'origin/exp/010-affectnet-train'.
Already up to date.
b218779 (HEAD -> exp/010-affectnet-train, origin/exp/010-affectnet-train) EXP-010: add executed AffectNet-train notebook
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


# Feature ảnh/text test (P0, gốc + lật)

In [3]:
import sys, glob, re, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
OOD = ["RAFDB", "FERPlus"]   # khác miền với AffectNet-train
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
CFG_NAMES = ["P0", "P0+flip"]

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
}
for E in EPOCHS_TEST:
    HEADS[f"EXP-004 e{E}"] = {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e{E}-model.pth" for s in SEEDS}
    HEADS[f"plain e{E}"] = {s: f"{OUT_DIR}/{job_of('plain', s)}-e{E}-model.pth" for s in SEEDS}
    HEADS[f"anchor e{E}"] = {s: f"{OUT_DIR}/{job_of('anchor', s)}-e{E}-model.pth" for s in SEEDS}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[ds] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", image_transform("raw"))
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        f0, f1 = [], []
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
        img[ds_name] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name][0].shape), classes[ds_name])

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|████████████████████████████████████████| 890M/890M [00:04<00:00, 211MiB/s]


RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']


In [4]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, ds_name, cfg, pn):
    f0, f1 = img[ds_name]
    i, t = ((f0 + f1) / 2 if cfg.endswith("+flip") else f0), txt[(ds_name, pn)]   # TTA: trung bình trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    r = metrics(y[ds_name], predict(W, ds_name, cfg, pn), N_CLS[ds_name])
                    rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn,
                                 "UAR": r["UAR"], "WAR": r["WAR"],
                                 **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp010b_anchor_epochs-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

640 dòng → /content/drive/MyDrive/ExpCLIP_output/exp010


# Kiểm tra pipeline
Head e30 phải khớp số đã ghi (EXP-004 từ `test.py`, EXP-010 từ notebook EXP-010). Lệch ≤ ~0.4 là do fp16.

In [5]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004 e30|1|RAFDB": [58.73, 65.94], "EXP-004 e30|1|AffectNet7": [45.83, 45.83], "EXP-004 e30|1|AffectNet8": [39.6, 39.6], "EXP-004 e30|1|FERPlus": [46.14, 52.03], "EXP-004 e30|2|RAFDB": [57.07, 64.77], "EXP-004 e30|2|AffectNet7": [43.97, 43.97], "EXP-004 e30|2|AffectNet8": [38.75, 38.75], "EXP-004 e30|2|FERPlus": [45.63, 52.45], "EXP-004 e30|3|RAFDB": [58.26, 65.06], "EXP-004 e30|3|AffectNet7": [45.46, 45.46], "EXP-004 e30|3|AffectNet8": [39.07, 39.08], "EXP-004 e30|3|FERPlus": [45.34, 51.44]}
REF.update({"anchor e30|1|RAFDB": [60.28, 60.79], "anchor e30|1|AffectNet7": [44.26, 44.26],
            "anchor e30|1|AffectNet8": [38.23, 38.23], "anchor e30|1|FERPlus": [44.63, 50.35]})   # test.py, EXP-010

chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == "P0") & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))

28/28 cấu hình so được | max |ΔUAR| = 0.35, max |ΔWAR| = 0.09
                     key  dUAR  dWAR
       EXP-000|3|FERPlus -0.35  0.06
      anchor e30|1|RAFDB -0.16  0.03
 anchor e30|1|AffectNet7 -0.09 -0.09
         EXP-000|2|RAFDB  0.09 -0.00
EXP-004 e30|1|AffectNet8 -0.08 -0.08
EXP-004 e30|2|AffectNet7  0.06  0.06


# Bảng 1 — UAR (mean ± std qua seed), Ensemble Five
`OOD2` = trung bình RAFDB + FERPlus.

In [6]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

u = res.pivot_table(index=["model", "seed", "prompt", "config"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u[DATASETS].mean(1)
u["OOD2"] = u[OOD].mean(1)

def get(m, cfg, pn=BASE_PROMPT):
    return u.loc[m].xs(pn, level="prompt").xs(cfg, level="config")   # index: seed

for pn in PROMPT_NAMES:
    for c in CFG_NAMES:
        t = pd.DataFrame({m: get(m, c, pn).agg(fmt) for m in HEADS if m in u.index.get_level_values(0)}).T
        print(f"--- {pn} @{c}\n{t.to_string()}\n")

--- Expression Ensemble Five @P0
dataset             RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4          OOD2
Paper               57.90         44.23         38.95         43.17         46.06         50.54
EXP-000      58.10 ± 0.69  44.14 ± 0.55  38.82 ± 0.60  41.42 ± 1.02  45.62 ± 0.40  49.76 ± 0.28
EXP-004 e5   56.45 ± 3.05  43.26 ± 0.81  37.98 ± 0.73  41.09 ± 2.35  44.70 ± 1.65  48.77 ± 2.70
plain e5     59.82 ± 1.56  44.21 ± 0.43  39.34 ± 0.30  46.88 ± 0.97  47.56 ± 0.29  53.35 ± 0.33
anchor e5    60.13 ± 2.31  43.90 ± 0.98  38.73 ± 0.60  43.41 ± 1.87  46.54 ± 0.46  51.77 ± 1.34
EXP-004 e10  58.13 ± 2.31  44.01 ± 0.46  38.42 ± 0.42  42.52 ± 1.35  45.77 ± 1.06  50.32 ± 1.83
plain e10    59.61 ± 1.25  43.79 ± 0.43  38.93 ± 0.24  47.93 ± 0.64  47.56 ± 0.32  53.77 ± 0.32
anchor e10   60.90 ± 2.09  43.75 ± 0.88  38.42 ± 0.60  43.60 ± 1.75  46.67 ± 0.25  52.25 ± 0.83
EXP-004 e20  58.24 ± 1.18  44.71 ± 0.70  38.79 ± 0.25  44.45 ± 0.97  46.55 ± 0.72  51.34 ± 1.07
plain e

# Bảng 2 — Chênh lệch theo cặp seed (Ensemble Five, P0) + tiêu chí

In [7]:
def compare(a, b, cfg="P0", pn=BASE_PROMPT):
    d = get(a, cfg, pn) - get(b, cfg, pn)
    return d.agg(fmt), d.mean()

t, verdict = {}, {}
for E in EPOCHS_TEST:
    for name, (a, b) in {f"anchor e{E} − plain e5": (f"anchor e{E}", "plain e5"),
                         f"anchor e{E} − EXP-004 e{E}": (f"anchor e{E}", f"EXP-004 e{E}"),
                         f"anchor e{E} − plain e{E}": (f"anchor e{E}", f"plain e{E}")}.items():
        try:
            t[name], mean = compare(a, b)
        except KeyError:
            print("thiếu checkpoint:", name); continue
        if name.endswith("plain e5"):
            ok = mean["Mean4"] >= 0.5 and (mean[DATASETS] >= -0.5).all()
            verdict[name] = (f"Δ Mean4 {mean['Mean4']:+.2f}, min Δ dataset {mean[DATASETS].min():+.2f}, "
                             f"Δ OOD2 {mean['OOD2']:+.2f} → {'ĐẠT' if ok else 'KHÔNG đạt'}")
for name in ["plain e5 − EXP-000", "EXP-004 e5 − EXP-000", "EXP-004 e30 − EXP-000"]:
    a, b = name.split(" − ")
    t[name], _ = compare(a, b)
print(pd.DataFrame(t).T.to_string(), "\n")
for k, v in verdict.items():
    print(f"{k}: {v}")

dataset                          RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4          OOD2
anchor e5 − plain e5       0.30 ± 1.30  -0.31 ± 0.62  -0.61 ± 0.30  -3.48 ± 1.17  -1.02 ± 0.38  -1.59 ± 1.18
anchor e5 − EXP-004 e5     3.68 ± 0.76   0.64 ± 0.66   0.75 ± 0.79   2.31 ± 3.27   1.84 ± 1.22   3.00 ± 2.00
anchor e10 − plain e5      1.07 ± 0.76  -0.46 ± 0.52  -0.92 ± 0.35  -3.28 ± 0.88  -0.90 ± 0.12  -1.10 ± 0.67
anchor e10 − EXP-004 e10   2.76 ± 0.22  -0.26 ± 0.44  -0.01 ± 0.81   1.09 ± 2.88   0.90 ± 0.81   1.93 ± 1.55
anchor e10 − plain e10     1.28 ± 1.26  -0.04 ± 0.59  -0.51 ± 0.39  -4.33 ± 1.23  -0.90 ± 0.23  -1.52 ± 0.94
anchor e20 − plain e5      0.52 ± 1.11  -0.36 ± 0.51  -1.17 ± 0.45  -2.88 ± 0.05  -0.97 ± 0.13  -1.18 ± 0.54
anchor e20 − EXP-004 e20   2.11 ± 1.17  -0.87 ± 0.24  -0.62 ± 0.68  -0.44 ± 1.91   0.05 ± 0.53   0.83 ± 1.39
anchor e20 − plain e20     2.07 ± 1.28   0.21 ± 0.95  -0.65 ± 0.82  -3.67 ± 0.49  -0.51 ± 0.24  -0.80 ± 0.53
anchor e30 − plain 

# Bảng 3 — Accuracy từng lớp (Ensemble Five, P0, mean seed)

In [8]:
for ds_name in DATASETS:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    t = {}
    for m in ["EXP-000", "plain e5"] + [f"anchor e{E}" for E in EPOCHS_TEST] + ["EXP-004 e30"]:
        r = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT) & (res.config == "P0")]
        t[m] = r[cols].mean().values
    t = pd.DataFrame(t, index=classes[ds_name]).T
    print(f"--- {ds_name}\n{t.round(1).to_string()}\n")

--- RAFDB
             Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
EXP-000           87.2     45.7     45.4   88.9      60.3     50.0  29.3
plain e5          81.8     55.6     29.7   82.5      63.2     74.4  31.5
anchor e5         62.0     46.5     49.1   84.4      70.3     69.0  39.6
anchor e10        64.3     53.3     50.0   84.4      72.8     65.8  35.6
anchor e20        63.0     59.8     47.5   85.2      76.7     62.3  27.9
anchor e30        61.3     62.2     44.9   85.4      78.2     60.0  25.2
EXP-004 e30       89.3     52.6     32.9   91.4      70.2     42.7  27.0

--- AffectNet7
             Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
EXP-000           75.7     45.8     24.3   48.1      33.7     49.0  32.4
plain e5          74.2     52.5     21.1   38.0      32.1     63.4  28.1
anchor e5         57.5     47.7     27.7   37.9      41.7     59.1  35.6
anchor e10        56.5     49.2     30.8   37.4      41.9     57.2  33.3
anchor e20        54.8   

# Lưu JSON (cùng định dạng `research_log/results/`)

In [9]:
out = {"source": "notebooks/EXP_010b_anchor_epochs.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-010b-anchor-epochs.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-010b-anchor-epochs.json")

640 dòng → /content/drive/MyDrive/ExpCLIP_output/exp010/EXP-010b-anchor-epochs.json
